In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, matthews_corrcoef
from kneed import KneeLocator

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

In [2]:
phase2_df1=pd.read_csv('/home/imokhtatif/.vscode-server/Chlamy_Project_v2-main/Data/2025_5_phase2.csv',low_memory=False)

In [3]:
from scipy import interpolate
from scipy.stats import rankdata

def normalize_quantiles(A, ties=True):
    A = np.asarray(A, dtype=np.float64)
    n_rows, n_cols = A.shape
    if n_cols == 1:
        return A.copy()

    i = np.linspace(0, 1, n_rows)
    S = np.full((n_rows, n_cols), np.nan)
    nobs = np.zeros(n_cols, dtype=int)
    sort_idx = []

    for j in range(n_cols):
        col = A[:, j]
        not_nan = ~np.isnan(col)
        x = col[not_nan]
        nobs[j] = len(x)
        sort_order = np.argsort(x)
        sorted_x = x[sort_order]

        if nobs[j] < n_rows:
            f = interpolate.interp1d(np.linspace(0, 1, nobs[j]), sorted_x,
                                     bounds_error=False, fill_value="extrapolate")
            S[:, j] = f(i)
        else:
            S[:, j] = sorted_x

        sort_idx.append(np.argsort(np.argsort(col[not_nan])))

    m = np.nanmean(S, axis=1)
    A_out = np.full_like(A, np.nan)

    for j in range(n_cols):
        col = A[:, j]
        not_nan = ~np.isnan(col)

        if ties:
            r = rankdata(col[not_nan], method='average')
            quant_pos = (r - 1) / (nobs[j] - 1)
            f = interpolate.interp1d(i, m, bounds_error=False, fill_value="extrapolate")
            A_out[not_nan, j] = f(quant_pos)
        else:
            ranks = sort_idx[j]
            A_out[not_nan, j] = m[ranks.astype(int)]

    return A_out

In [4]:
# Step 1: Filter data for the three plates
plates = ['33v1', '33v2', '33v3']
df_30v =phase2_df1[phase2_df1['plate'].isin(plates)]

# Step 2: Count rows per (plate, mutant_ID, mutated_genes, light_regime)
group_counts = (
    df_30v.groupby(['plate', 'light_regime', 'mutant_ID', 'mutated_genes'])
    .size()
    .reset_index(name='count')
)

# Step 3: For each plate and light_regime, count how many mutants had 1, 2, ... rows
summary = (
    group_counts.groupby(['light_regime','plate', 'count'])
    .size()
    .reset_index(name='n_mutants')
)

# Optional: Sort for easier reading
summary = summary.sort_values(by=['light_regime','plate', 'count'])

# Show result
summary

,light_regime,plate,count,n_mutants
0,10min-10min,33v1,1,361
1,10min-10min,33v1,7,1
2,10min-10min,33v2,1,361
3,10min-10min,33v2,7,1
4,1min-1min,33v1,1,360
5,1min-1min,33v1,7,1
6,1min-1min,33v2,1,361
7,1min-1min,33v2,7,1
8,1min-1min,33v3,1,361
9,1min-1min,33v3,7,1


In [5]:
def quantile_normalize_light_regime(df, light_regime, plates, y2_cols, tie_handling=True):

    # Filter data
    subset_df = df[(df['light_regime'] == light_regime) & (df['plate'].isin(plates))].copy()
    df_normalized = subset_df.copy()

    for timepoint in y2_cols:
        position_values = []
        valid_plate_indices = {}

        for plate in plates:
            plate_df = subset_df[subset_df['plate'] == plate].copy()

            wt_rows = plate_df[plate_df['mutant_ID'] == 'WT'].copy()
            non_wt_rows = plate_df[plate_df['mutant_ID'] != 'WT'].copy()

            wt_rows = wt_rows.sort_values(['mutant_ID', 'mutated_genes', 'well_id'])
            non_wt_rows = non_wt_rows.sort_values(['mutant_ID', 'mutated_genes'])

            sorted_df = pd.concat([wt_rows, non_wt_rows], axis=0)
            # print('plate',plate ,wt_rows[timepoint].values)
            values = sorted_df[timepoint].values
            index = sorted_df.index.values

            position_values.append(values)
            valid_plate_indices[plate] = index

        # Validate shape
        lengths = [len(v) for v in position_values]
        if len(set(lengths)) != 1:
            raise ValueError(f"Length mismatch at {timepoint}: {lengths}")

        matrix = np.column_stack(position_values)
        normalized_matrix = normalize_quantiles(matrix, ties=tie_handling)

        # Write back
        for col_idx, plate in enumerate(plates):
            indices = valid_plate_indices[plate]
            df_normalized.loc[indices, timepoint] = normalized_matrix[:, col_idx]

    return df_normalized

## plate 33 20h hl

In [6]:

plates = ['33v1', '33v2', '33v3']
y2_cols = [f'y2_{i}' for i in range(1, 45)]

# Run normalization
phase2_33_20h_ML_normalized = quantile_normalize_light_regime(
    df=phase2_df1,
    light_regime='20h_ML',
    plates=plates,
    y2_cols=y2_cols
)

# View a few columns
phase2_33_20h_ML_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_35,y2_36,y2_37,y2_38,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44
36386,33v1,LMJ.RY0402.144439,Cre16.g664150,P23,0.366324,0.380732,0.408007,0.362953,0.387254,0.407195,...,0.325034,0.368269,0.380102,0.348049,0.367857,0.339861,0.376763,0.352939,0.331811,0.348548
36387,33v1,LMJ.RY0402.218008,Cre08.g374250,F04,0.294490,0.299321,0.349117,0.317049,0.346981,0.338652,...,0.291174,0.276721,0.274852,0.297054,0.292100,0.291702,0.293066,0.296127,0.289060,0.261035
36388,33v1,LMJ.RY0402.254684,Cre12.g488400,F05,0.281579,0.343913,0.304023,0.340504,0.285892,0.343264,...,0.276459,0.248265,0.268141,0.225018,0.268026,0.250685,0.290248,0.283163,0.250840,0.259017
36389,33v1,LMJ.RY0402.041443,Cre13.g606101,F06,0.419119,0.465200,0.474539,0.489911,0.522166,0.493325,...,0.458265,0.462451,0.456148,0.471790,0.469231,0.434919,0.456615,0.484363,0.459819,0.478419
36390,33v1,LMJ.RY0402.078851,Cre14.g617800,F07,0.319794,0.356236,0.358916,0.362306,0.380354,0.367137,...,0.309173,0.291462,0.299367,0.281524,0.298822,0.305597,0.300596,0.296558,0.312438,0.279488
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42631,33v3,LMJ.RY0402.232230,Cre03.g207351,K22,0.374546,0.389392,0.401079,0.407105,0.385658,0.425665,...,0.314783,0.382743,0.340452,0.373971,0.317362,0.320221,0.322292,0.369575,0.346897,0.310076
42632,33v3,LMJ.RY0402.188401,Cre06.g259950,K23,0.455982,0.496961,0.467728,0.425861,0.412862,0.532244,...,0.381062,0.374621,0.424302,0.407681,0.415149,0.442278,0.414535,0.404941,0.424591,0.429507
42633,33v3,LMJ.RY0402.228944,"Cre13.g588250,Cre10.g426950",K14,0.391667,0.410482,0.419641,0.424934,0.403374,0.435623,...,0.318031,0.356788,0.371363,0.334292,0.319677,0.335422,0.306858,0.319047,0.316144,0.333707
42634,33v3,LMJ.RY0402.178705,Cre14.g617800,A03,0.390686,0.403063,0.336433,0.357343,0.385979,0.399167,...,0.368004,0.359208,0.354512,0.352391,0.356627,0.342730,0.364154,0.338552,0.330525,0.357188


In [7]:
plates = ['33v1', '33v2', '33v3']
phase2_33_20h_ML= phase2_df1[(phase2_df1['light_regime'] == '20h_ML') & (phase2_df1['plate'].isin(plates))].copy()
phase2_33_20h_ML[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_35,y2_36,y2_37,y2_38,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44
36386,33v1,LMJ.RY0402.144439,Cre16.g664150,P23,0.345114,0.354301,0.386892,0.326514,0.357129,0.380091,...,0.299781,0.334593,0.349068,0.309448,0.334668,0.306481,0.339810,0.316652,0.300371,0.315348
36387,33v1,LMJ.RY0402.218008,Cre08.g374250,F04,0.273273,0.278630,0.329464,0.274483,0.319721,0.308465,...,0.261336,0.244223,0.246838,0.254579,0.260691,0.254655,0.252516,0.258175,0.256163,0.226815
36388,33v1,LMJ.RY0402.254684,Cre12.g488400,F05,0.264855,0.317631,0.288894,0.303992,0.254310,0.312619,...,0.245306,0.214355,0.243104,0.175870,0.235093,0.223602,0.251022,0.244951,0.217047,0.225916
36389,33v1,LMJ.RY0402.041443,Cre13.g606101,F06,0.393244,0.439130,0.452708,0.456028,0.485532,0.458676,...,0.424522,0.422480,0.419169,0.430077,0.429671,0.398772,0.415091,0.440452,0.422812,0.434980
36390,33v1,LMJ.RY0402.078851,Cre14.g617800,F07,0.301796,0.330606,0.339349,0.325580,0.351643,0.338581,...,0.279906,0.255982,0.267195,0.243113,0.267176,0.273503,0.261649,0.258640,0.279417,0.245103
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42631,33v3,LMJ.RY0402.232230,Cre03.g207351,K22,0.396024,0.418497,0.429599,0.438434,0.413858,0.453645,...,0.336357,0.412087,0.366209,0.400176,0.355475,0.347746,0.351641,0.394235,0.373572,0.335166
42632,33v3,LMJ.RY0402.188401,Cre06.g259950,K23,0.481572,0.527381,0.497354,0.456512,0.439228,0.557838,...,0.401883,0.406221,0.448266,0.433616,0.443315,0.469758,0.444734,0.434406,0.459817,0.450126
42633,33v3,LMJ.RY0402.228944,"Cre13.g588250,Cre10.g426950",K14,0.419148,0.439186,0.451643,0.456189,0.427689,0.462564,...,0.339190,0.386749,0.393694,0.363092,0.356814,0.360856,0.338546,0.341128,0.348207,0.356625
42634,33v3,LMJ.RY0402.178705,Cre14.g617800,A03,0.417460,0.431703,0.367956,0.396826,0.414155,0.430241,...,0.382124,0.388257,0.377983,0.379674,0.390154,0.367399,0.390338,0.361966,0.358875,0.376540


## plate 33 20h ML

In [8]:

plates = ['33v1', '33v2', '33v3']
y2_cols = [f'y2_{i}' for i in range(1, 45)]

# Run normalization
phase2_33_20h_HL_normalized = quantile_normalize_light_regime(
    df=phase2_df1,
    light_regime='20h_HL',
    plates=plates,
    y2_cols=y2_cols
)
phase2_33_20h_HL_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_35,y2_36,y2_37,y2_38,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44
36018,33v1,LMJ.RY0402.168114,Cre01.g024800,A05,0.232493,0.175098,0.186078,0.235897,0.196988,0.205057,...,0.136252,0.137570,0.123050,0.146577,0.098014,0.135838,0.139061,0.162430,0.133587,0.110807
36019,33v1,LMJ.RY0402.145715,Cre06.g295600,L01,0.320134,0.299388,0.328903,0.259378,0.312357,0.262492,...,0.265846,0.221338,0.211878,0.247455,0.229525,0.256377,0.222816,0.146456,0.212314,0.242044
36020,33v1,LMJ.RY0402.242322,Cre04.g214097,K24,0.195151,0.331605,0.262441,0.252364,0.258735,0.265928,...,0.148874,0.134089,0.121561,0.224398,0.213517,0.156373,0.109206,0.053487,0.075415,0.092693
36021,33v1,LMJ.RY0402.230639,Cre07.g322550,K23,0.214256,0.224366,0.210108,0.211903,0.234289,0.192669,...,0.149512,0.144611,0.072408,0.149454,0.113892,0.128761,0.093740,0.131451,0.157438,0.095430
36022,33v1,LMJ.RY0402.191679,Cre10.g433450,K22,0.122905,0.179925,0.240725,0.230269,0.217499,0.228034,...,0.074123,0.166689,0.144734,0.153339,0.108168,0.135357,0.083877,0.152150,0.130968,0.117417
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42263,33v3,LMJ.RY0402.058624,Cre13.g588150,K18,0.178317,0.156145,0.164372,0.220933,0.148163,0.218468,...,0.066313,-0.022583,0.086659,0.083577,0.091818,0.055090,0.026512,0.038836,0.115184,0.041773
42264,33v3,LMJ.RY0402.182401,Cre12.g488400,K17,0.154378,0.166041,0.131112,0.136055,0.198994,0.117878,...,0.072048,0.101766,0.065427,0.068193,0.074974,0.058349,0.093325,0.087709,0.080515,0.057167
42265,33v3,LMJ.RY0402.201845,"Cre03.g170001,Cre04.g223100",K16,0.203631,0.159792,0.223745,0.272087,0.255956,0.281877,...,0.174170,0.154852,0.151081,0.151296,0.176755,0.168606,0.164149,0.150905,0.130968,0.132550
42266,33v3,LMJ.RY0402.132166,Cre07.g332300,K15,0.267167,0.302104,0.231914,0.224901,0.264299,0.254023,...,0.124735,0.089331,0.085149,0.124545,0.106404,0.108473,0.076698,0.070993,0.058512,0.093309


In [9]:
plates = ['33v1', '33v2', '33v3']
phase2_33_20h_HL= phase2_df1[(phase2_df1['light_regime'] == '20h_HL') & (phase2_df1['plate'].isin(plates))].copy()
phase2_33_20h_HL[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_35,y2_36,y2_37,y2_38,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44
36018,33v1,LMJ.RY0402.168114,Cre01.g024800,A05,0.212197,0.162335,0.167037,0.213647,0.174096,0.179443,...,0.134432,0.130809,0.108168,0.135116,0.091057,0.124378,0.128819,0.146431,0.121531,0.102198
36019,33v1,LMJ.RY0402.145715,Cre06.g295600,L01,0.311783,0.282077,0.318055,0.239704,0.294371,0.238905,...,0.253942,0.208777,0.191580,0.263546,0.214781,0.238050,0.220711,0.129309,0.199942,0.234229
36020,33v1,LMJ.RY0402.242322,Cre04.g214097,K24,0.177755,0.311075,0.248009,0.232861,0.237220,0.242318,...,0.146324,0.127417,0.107169,0.211252,0.198400,0.147202,0.099884,0.034105,0.062690,0.087276
36021,33v1,LMJ.RY0402.230639,Cre07.g322550,K23,0.195666,0.207950,0.190658,0.190901,0.206683,0.171224,...,0.147194,0.136155,0.057810,0.138507,0.106140,0.117896,0.086059,0.114124,0.148796,0.088146
36022,33v1,LMJ.RY0402.191679,Cre10.g433450,K22,0.107793,0.168570,0.226283,0.208181,0.191216,0.202407,...,0.075400,0.155583,0.131910,0.142040,0.101768,0.124100,0.074385,0.135467,0.118502,0.108870
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42263,33v3,LMJ.RY0402.058624,Cre13.g588150,K18,0.175041,0.154808,0.163104,0.229138,0.152832,0.220982,...,0.042825,-0.046637,0.075372,0.073627,0.072863,0.044281,0.010453,0.026938,0.101620,0.032057
42264,33v3,LMJ.RY0402.182401,Cre12.g488400,K17,0.148684,0.164182,0.127378,0.135115,0.203784,0.116885,...,0.049761,0.082316,0.059130,0.062543,0.058115,0.046619,0.070679,0.073344,0.069943,0.046273
42265,33v3,LMJ.RY0402.201845,"Cre03.g170001,Cre04.g223100",K16,0.200044,0.158253,0.223075,0.285237,0.259789,0.290238,...,0.153983,0.135057,0.138862,0.139152,0.161126,0.151781,0.143901,0.136964,0.118357,0.121050
42266,33v3,LMJ.RY0402.132166,Cre07.g332300,K15,0.271903,0.313255,0.230977,0.234458,0.269494,0.258691,...,0.105504,0.069427,0.073851,0.114767,0.086719,0.093079,0.054097,0.056003,0.049348,0.078085


## plate 33 2h-2h

In [11]:
plates = ['33v1', '33v2', '33v3']
y2_cols = [f'y2_{i}' for i in range(1, 49)]

plates_of_interest = plates  # or your list of plates
subset = phase2_df1[
    (phase2_df1['light_regime'] == '2h-2h') &
    (phase2_df1['plate'].isin(plates_of_interest))
]
mutants_by_plate = {
    plate: set(subset[subset['plate'] == plate]['mutant_ID'])
    for plate in plates_of_interest
}
common_mutants = set.intersection(*mutants_by_plate.values())
filtered_df = subset[subset['mutant_ID'].isin(common_mutants)].copy()


# Run normalization
phase2_33_2h_2h_normalized = quantile_normalize_light_regime(
    df=filtered_df,
    light_regime='2h-2h',
    plates=plates,
    y2_cols=y2_cols
)
phase2_33_2h_2h_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44,y2_45,y2_46,y2_47,y2_48
36754,33v1,LMJ.RY0402.173729,Cre08.g358571,F02,0.219617,0.222269,0.166536,0.223123,0.230302,0.241887,...,0.618001,0.614415,0.122658,0.163947,0.133087,0.120268,0.588040,0.585378,0.601702,0.615662
36755,33v1,LMJ.RY0402.043444,Cre02.g105350,F03,0.200509,0.134778,0.191236,0.241458,0.198658,0.238733,...,0.621739,0.627841,0.152343,0.091214,0.169909,0.168852,0.609599,0.621689,0.609606,0.640367
36756,33v1,LMJ.RY0402.218008,Cre08.g374250,F04,0.125892,0.168142,0.175836,0.133119,0.178437,0.169977,...,0.515474,0.544019,0.112446,0.099420,0.106603,0.134480,0.527405,0.535438,0.543304,0.525347
36757,33v1,LMJ.RY0402.254684,Cre12.g488400,F05,0.148235,0.182811,0.197105,0.220222,0.145104,0.158350,...,0.583979,0.576153,0.140304,0.096941,0.103971,0.095560,0.553936,0.568040,0.568336,0.585197
36758,33v1,LMJ.RY0402.041443,Cre13.g606101,F06,0.249637,0.212099,0.280066,0.259851,0.242222,0.265617,...,0.675297,0.679803,0.262187,0.267113,0.275046,0.233144,0.684973,0.683314,0.693315,0.686474
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42998,33v3,LMJ.RY0402.065715,"Cre01.g024800,Cre01.g040250",K20,0.236973,0.190015,0.228429,0.238140,0.260638,0.250930,...,0.673141,0.652093,0.220057,0.123429,0.186664,0.169649,0.615887,0.649587,0.647036,0.639480
42999,33v3,LMJ.RY0402.191955,Cre09.g397734,K21,0.170404,0.130322,0.163937,0.177255,0.196626,0.190535,...,0.597409,0.559084,0.075927,0.142965,0.068321,0.080043,0.550590,0.553933,0.576275,0.579247
43000,33v3,LMJ.RY0402.232230,Cre03.g207351,K22,0.215326,0.241796,0.170449,0.174605,0.234899,0.212390,...,0.580909,0.577065,0.167758,0.150194,0.149603,0.166945,0.561239,0.589200,0.587623,0.588680
43001,33v3,LMJ.RY0402.188401,Cre06.g259950,K23,0.302504,0.312005,0.364537,0.313834,0.252775,0.219715,...,0.690020,0.694089,0.106367,0.250514,0.160567,0.123750,0.658340,0.670789,0.683925,0.687141


In [12]:
plates = ['33v1', '33v2', '33v3']
phase2_33_2h_2h= phase2_df1[(phase2_df1['light_regime'] == '2h-2h') & (phase2_df1['plate'].isin(plates))].copy()
phase2_33_2h_2h[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44,y2_45,y2_46,y2_47,y2_48
36754,33v1,LMJ.RY0402.173729,Cre08.g358571,F02,0.214909,0.215056,0.166536,0.215235,0.227622,0.241277,...,0.616272,0.613533,0.111730,0.158248,0.115028,0.121907,0.586004,0.584434,0.598786,0.611918
36755,33v1,LMJ.RY0402.043444,Cre02.g105350,F03,0.198456,0.122497,0.191463,0.232639,0.194403,0.237260,...,0.619140,0.625163,0.142895,0.088555,0.150679,0.171521,0.607669,0.618404,0.605708,0.634791
36756,33v1,LMJ.RY0402.218008,Cre08.g374250,F04,0.116131,0.158891,0.175769,0.124660,0.173455,0.166809,...,0.518251,0.536322,0.100419,0.097443,0.088449,0.138851,0.521987,0.531841,0.536254,0.522111
36757,33v1,LMJ.RY0402.254684,Cre12.g488400,F05,0.140672,0.175759,0.197335,0.211622,0.139808,0.154643,...,0.581335,0.570267,0.127539,0.094533,0.086293,0.097698,0.551935,0.563613,0.562722,0.581516
36758,33v1,LMJ.RY0402.041443,Cre13.g606101,F06,0.247894,0.204386,0.279919,0.248611,0.242665,0.267874,...,0.672810,0.678952,0.248785,0.265456,0.254893,0.231568,0.682986,0.683174,0.690097,0.683974
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42998,33v3,LMJ.RY0402.065715,"Cre01.g024800,Cre01.g040250",K20,0.245783,0.209503,0.233606,0.252678,0.267426,0.256562,...,0.668056,0.652077,0.213968,0.124276,0.195925,0.166140,0.617601,0.649798,0.648552,0.645247
42999,33v3,LMJ.RY0402.191955,Cre09.g397734,K21,0.185104,0.153471,0.174335,0.197800,0.201161,0.200566,...,0.595712,0.562960,0.079283,0.147351,0.092294,0.079296,0.553549,0.555561,0.580215,0.585031
43000,33v3,LMJ.RY0402.232230,Cre03.g207351,K22,0.226999,0.259597,0.181488,0.196048,0.238086,0.221545,...,0.580016,0.578645,0.169703,0.156623,0.162123,0.163317,0.563217,0.589225,0.588898,0.593375
43001,33v3,LMJ.RY0402.188401,Cre06.g259950,K23,0.305318,0.324677,0.369460,0.324740,0.258075,0.229303,...,0.684730,0.693800,0.106403,0.244418,0.172509,0.122719,0.655505,0.669084,0.685540,0.687244


## plate 33 10min-10min

In [13]:
plates = ['33v1', '33v2']
y2_cols = [f'y2_{i}' for i in range(1, 85)]

# Run normalization
phase2_33_10min_10min_normalized = quantile_normalize_light_regime(
    df=phase2_df1,
    light_regime='10min-10min',
    plates=plates,
    y2_cols=y2_cols
)
phase2_33_10min_10min_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_75,y2_76,y2_77,y2_78,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84
37489,33v1,LMJ.RY0402.183511,Cre13.g585250,P22,0.159143,0.582719,0.225440,0.539931,0.137514,0.590330,...,0.155348,0.545093,0.579909,0.166793,0.159781,0.532657,0.558485,0.118588,0.099542,0.542540
37490,33v1,LMJ.RY0402.049580,"Cre12.g496100,Cre12.g496150 & Cre12.g496100,Cr...",K21,0.133411,0.516761,0.172386,0.559886,0.183097,0.513250,...,0.036652,0.519653,0.500975,0.093574,0.037664,0.541082,0.526652,0.071647,0.060336,0.489611
37491,33v1,LMJ.RY0402.100101,Cre02.g095058,K20,0.175579,0.596300,0.181273,0.595800,0.159651,0.599228,...,0.174922,0.579806,0.574534,0.105637,0.169599,0.597961,0.617029,0.151404,0.144726,0.577628
37492,33v1,LMJ.RY0402.254836,Cre04.g215150,K18,0.286027,0.632487,0.252395,0.598250,0.267886,0.622011,...,0.257474,0.565458,0.570077,0.169875,0.179922,0.580764,0.572555,0.234240,0.187890,0.555633
37493,33v1,LMJ.RY0402.046384,Cre01.g012650,K17,0.077611,0.507912,0.120236,0.512644,0.048157,0.496590,...,0.050063,0.470956,0.451743,0.076828,0.035658,0.472607,0.500176,0.083939,0.064431,0.459829
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40791,33v2,LMJ.RY0402.088794,Cre12.g526850,P21,0.226568,0.563916,0.152793,0.602778,0.200750,0.586589,...,0.096885,0.611118,0.602224,0.146091,0.125262,0.573667,0.605310,0.102990,0.162757,0.576878
40792,33v2,LMJ.RY0402.181834,Cre02.g078939,P20,0.236283,0.566775,0.164535,0.551973,0.140937,0.590330,...,0.146735,0.558814,0.548114,0.168905,0.101994,0.582812,0.566723,-0.006288,0.090302,0.549149
40793,33v2,LMJ.RY0402.065793,Cre13.g588600,A02,0.141017,0.518284,0.176228,0.530826,0.141088,0.513250,...,0.113031,0.531116,0.520096,0.086796,0.103766,0.505506,0.499049,0.150762,0.127171,0.494289
40794,33v2,LMJ.RY0402.193585,"Cre17.g746397,Cre06.g275950",P24,0.315670,0.622860,0.129297,0.639137,0.290181,0.615485,...,0.141352,0.673289,0.615893,0.216501,0.135144,0.664026,0.616492,0.187026,0.157219,0.639362


In [14]:
plates = ['33v1', '33v2', '33v3']
phase2_33_10min_10min= phase2_df1[(phase2_df1['light_regime'] == '10min-10min') & (phase2_df1['plate'].isin(plates))].copy()
phase2_33_10min_10min[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_75,y2_76,y2_77,y2_78,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84
37489,33v1,LMJ.RY0402.183511,Cre13.g585250,P22,0.149867,0.585443,0.221655,0.543200,0.133046,0.589225,...,0.147467,0.544232,0.580586,0.157465,0.152082,0.534962,0.558348,0.108602,0.091559,0.544387
37490,33v1,LMJ.RY0402.049580,"Cre12.g496100,Cre12.g496150 & Cre12.g496100,Cr...",K21,0.125112,0.517666,0.165614,0.559437,0.174925,0.512573,...,0.031610,0.519189,0.503631,0.089912,0.023823,0.542790,0.525083,0.061339,0.051347,0.493508
37491,33v1,LMJ.RY0402.100101,Cre02.g095058,K20,0.167206,0.596223,0.176136,0.593549,0.157054,0.598741,...,0.169292,0.578644,0.574402,0.100158,0.162401,0.599080,0.616940,0.144030,0.133952,0.581032
37492,33v1,LMJ.RY0402.254836,Cre04.g215150,K18,0.272921,0.634849,0.246445,0.596631,0.254239,0.622508,...,0.254678,0.562333,0.571157,0.160682,0.175743,0.582987,0.573145,0.224704,0.180818,0.557561
37493,33v1,LMJ.RY0402.046384,Cre01.g012650,K17,0.066095,0.507687,0.124361,0.512796,0.048444,0.495316,...,0.043965,0.468358,0.460759,0.070258,0.021612,0.477562,0.500107,0.072373,0.054934,0.465022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40791,33v2,LMJ.RY0402.088794,Cre12.g526850,P21,0.239637,0.561796,0.155073,0.605066,0.209545,0.589501,...,0.101334,0.613778,0.601397,0.153440,0.134593,0.573097,0.605552,0.113960,0.171665,0.573791
40792,33v2,LMJ.RY0402.181834,Cre02.g078939,P20,0.248157,0.564367,0.170348,0.551906,0.144217,0.591434,...,0.153255,0.560771,0.545166,0.178701,0.108714,0.579657,0.567337,0.021844,0.098138,0.548124
40793,33v2,LMJ.RY0402.065793,Cre13.g588600,A02,0.149290,0.518045,0.182035,0.529671,0.144352,0.513928,...,0.117279,0.533371,0.516545,0.090605,0.110528,0.503062,0.499700,0.157956,0.134654,0.490616
40794,33v2,LMJ.RY0402.193585,"Cre17.g746397,Cre06.g275950",P24,0.317458,0.624196,0.127328,0.641191,0.300306,0.615621,...,0.147401,0.675676,0.613696,0.228051,0.143895,0.662929,0.617017,0.195498,0.166297,0.639677


## plate 33 1min-1min

In [16]:
plates = ['33v1', '33v2']
y2_cols = [f'y2_{i}' for i in range(1, 89)]


plates_of_interest = plates  # or your list of plates
subset = phase2_df1[
    (phase2_df1['light_regime'] == '1min-1min') &
    (phase2_df1['plate'].isin(plates_of_interest))
]
mutants_by_plate = {
    plate: set(subset[subset['plate'] == plate]['mutant_ID'])
    for plate in plates_of_interest
}
common_mutants = set.intersection(*mutants_by_plate.values())
filtered_df = subset[subset['mutant_ID'].isin(common_mutants)].copy()

# Run normalization
phase2_33_1min_1min_normalized = quantile_normalize_light_regime(
    df=filtered_df,
    light_regime='1min-1min',
    plates=plates,
    y2_cols=y2_cols
)
phase2_33_1min_1min_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
35651,33v1,LMJ.RY0402.227435,Cre12.g538850,A03,0.239812,0.583216,0.245523,0.567388,0.234619,0.554107,...,0.235659,0.581852,0.217723,0.533113,0.211700,0.526367,0.218025,0.520300,0.235357,0.533151
35652,33v1,LMJ.RY0402.041443,Cre13.g606101,F06,0.266607,0.585140,0.304589,0.600383,0.290567,0.574162,...,0.269781,0.593231,0.238206,0.582329,0.217283,0.570622,0.237120,0.562494,0.243726,0.549718
35653,33v1,LMJ.RY0402.078851,Cre14.g617800,F07,0.132614,0.480566,0.118539,0.488552,0.157273,0.454647,...,0.124537,0.445772,0.118865,0.464730,0.115959,0.462483,0.108511,0.436111,0.084631,0.432158
35654,33v1,LMJ.RY0402.114699,Cre17.g720261,F08,0.160941,0.557945,0.190902,0.519552,0.232430,0.544294,...,0.178621,0.514167,0.154614,0.499572,0.169681,0.507653,0.153187,0.500504,0.168676,0.513102
35655,33v1,LMJ.RY0402.061439,Cre12.g528250,F09,0.224406,0.546138,0.189555,0.499126,0.215194,0.512245,...,0.149133,0.483602,0.147756,0.490953,0.152498,0.475544,0.140758,0.473528,0.133261,0.490012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38953,33v2,LMJ.RY0402.144439,Cre16.g664150,F03,0.198362,0.530293,0.158244,0.501255,0.183089,0.524820,...,0.138709,0.465103,0.129474,0.456155,0.122060,0.490887,0.116298,0.467565,0.100545,0.470918
38954,33v2,LMJ.RY0402.119746,Cre13.g606250,F12,0.151167,0.522405,0.190086,0.513651,0.155622,0.501220,...,0.129644,0.491785,0.106017,0.499572,0.169681,0.497051,0.158922,0.494021,0.132192,0.486522
38955,33v2,LMJ.RY0402.181834,Cre02.g078939,P20,0.188142,0.553775,0.182231,0.479960,0.187508,0.545406,...,0.131040,0.504573,0.205072,0.526389,0.141927,0.487769,0.205235,0.520300,0.210753,0.487202
38956,33v2,LMJ.RY0402.118852,"Cre08.g372100,Cre04.g214501",P23,0.270422,0.517459,0.229244,0.487218,0.264278,0.495347,...,0.081695,0.469731,0.189355,0.509201,0.169012,0.519064,0.142039,0.494885,0.161652,0.451321


In [17]:
plates = ['33v1', '33v2', '33v3']
phase2_33_1min_1min= phase2_df1[(phase2_df1['light_regime'] == '1min-1min') & (phase2_df1['plate'].isin(plates))].copy()
phase2_33_1min_1min[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
35651,33v1,LMJ.RY0402.227435,Cre12.g538850,A03,0.246817,0.587317,0.256333,0.567975,0.246202,0.557925,...,0.243205,0.583613,0.220424,0.538117,0.214142,0.529889,0.222776,0.526620,0.241768,0.538201
35652,33v1,LMJ.RY0402.041443,Cre13.g606101,F06,0.268974,0.589576,0.313523,0.602306,0.299455,0.576863,...,0.277502,0.593291,0.241272,0.589532,0.217415,0.574783,0.241216,0.567799,0.250441,0.552978
35653,33v1,LMJ.RY0402.078851,Cre14.g617800,F07,0.140910,0.495791,0.140067,0.494139,0.173110,0.473425,...,0.139076,0.458732,0.128280,0.475144,0.127842,0.471085,0.120079,0.444270,0.094370,0.444140
35654,33v1,LMJ.RY0402.114699,Cre17.g720261,F08,0.169130,0.565577,0.204952,0.522413,0.242770,0.550809,...,0.189581,0.519407,0.160093,0.503125,0.173781,0.513134,0.160177,0.507249,0.176622,0.518298
35655,33v1,LMJ.RY0402.061439,Cre12.g528250,F09,0.230794,0.557483,0.203299,0.504321,0.226161,0.522824,...,0.161100,0.490249,0.153368,0.495289,0.159212,0.483966,0.148249,0.483059,0.140013,0.497865
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41895,33v3,LMJ.RY0402.182401,Cre12.g488400,K17,0.226979,0.572801,0.209042,0.537860,0.243425,0.525783,...,0.157536,0.502275,0.171955,0.497493,0.181973,0.506523,0.178785,0.508336,0.176259,0.499666
41896,33v3,LMJ.RY0402.065715,"Cre01.g024800,Cre01.g040250",K20,0.252561,0.578972,0.266566,0.538881,0.236870,0.580145,...,0.191592,0.560036,0.157678,0.540955,0.227308,0.543479,0.241890,0.530567,0.212253,0.554121
41897,33v3,LMJ.RY0402.132166,Cre07.g332300,K15,0.323856,0.632404,0.346622,0.611362,0.323266,0.601402,...,0.206425,0.546486,0.236785,0.539169,0.240456,0.583339,0.200190,0.547891,0.199570,0.531484
41898,33v3,LMJ.RY0402.228944,"Cre13.g588250,Cre10.g426950",K14,0.277836,0.589638,0.296988,0.560829,0.285820,0.542419,...,0.197080,0.505910,0.196010,0.513606,0.214002,0.509266,0.194965,0.537360,0.212104,0.491710


## plate 33 30s-30s

In [20]:
plates = ['33v2', '33v3']
y2_cols = [f'y2_{i}' for i in range(1, 89)]

plates_of_interest = plates  # or your list of plates
subset = phase2_df1[
    (phase2_df1['light_regime'] == '30s-30s') &
    (phase2_df1['plate'].isin(plates_of_interest))
]
mutants_by_plate = {
    plate: set(subset[subset['plate'] == plate]['mutant_ID'])
    for plate in plates_of_interest
}
common_mutants = set.intersection(*mutants_by_plate.values())
filtered_df = subset[subset['mutant_ID'].isin(common_mutants)].copy()


# Run normalization
phase2_33_30s_30s_normalized = quantile_normalize_light_regime(
    df=filtered_df,
    light_regime='30s-30s',
    plates=plates,
    y2_cols=y2_cols
)
phase2_33_30s_30s_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
40060,33v2,LMJ.RY0402.229127,"Cre10.g431250,Cre07.g315850",P19,0.375719,0.644068,0.347972,0.634991,0.362314,0.612144,...,0.280005,0.552108,0.315449,0.555909,0.332804,0.566869,0.322518,0.584942,0.331178,0.585758
40061,33v2,LMJ.RY0402.194210,Cre09.g389578,K22,0.390460,0.643532,0.319413,0.586903,0.345200,0.625696,...,0.258846,0.534925,0.246228,0.562553,0.272905,0.542555,0.258332,0.525154,0.275789,0.542664
40062,33v2,LMJ.RY0402.143718,Cre16.g648750,K21,0.371699,0.607168,0.286813,0.561873,0.312718,0.610087,...,0.199739,0.528081,0.241925,0.502713,0.258272,0.515750,0.221198,0.451973,0.238070,0.453997
40063,33v2,LMJ.RY0402.190807,Cre06.g302050,K20,0.220345,0.492239,0.178085,0.466576,0.164631,0.526580,...,0.178293,0.438127,0.170921,0.459117,0.167392,0.487912,0.082945,0.429598,0.194872,0.453179
40064,33v2,LMJ.RY0402.097352,Cre05.g236700,K19,0.140362,0.515648,0.166163,0.435761,0.165093,0.471948,...,0.107269,0.450082,0.163811,0.428225,0.196487,0.469418,0.185086,0.444277,0.170865,0.441701
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43365,33v3,LMJ.RY0402.174457,Cre17.g716050,F05,0.196851,0.546588,0.236413,0.502449,0.232549,0.483724,...,0.214716,0.498147,0.213017,0.495331,0.214786,0.480137,0.220105,0.450267,0.187684,0.451293
43366,33v3,LMJ.RY0402.156099,Cre16.g664850,F04,0.168135,0.488045,0.166536,0.467605,0.176403,0.486950,...,0.107269,0.432969,0.095932,0.445036,0.119082,0.422759,0.132549,0.413814,0.105287,0.416730
43367,33v3,LMJ.RY0402.115726,Cre14.g632750,F03,0.269423,0.589196,0.280190,0.573142,0.276206,0.571116,...,0.229437,0.548233,0.226583,0.527976,0.220276,0.532708,0.223741,0.515675,0.255209,0.552414
43368,33v3,LMJ.RY0402.229127,"Cre10.g431250,Cre07.g315850",F02,0.415171,0.666859,0.402587,0.628298,0.348444,0.632304,...,0.268293,0.605797,0.350215,0.574304,0.318670,0.568485,0.321835,0.564387,0.321735,0.598437


In [21]:
plates = ['33v1', '33v2', '33v3']
phase2_33_30s_30s= phase2_df1[(phase2_df1['light_regime'] == '30s-30s') & (phase2_df1['plate'].isin(plates))].copy()
phase2_33_30s_30s[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
37122,33v1,LMJ.RY0402.165552,Cre17.g735021,A07,0.172429,0.496955,0.179787,0.496948,0.182492,0.489767,...,0.156992,0.463003,0.119887,0.467151,0.154561,0.487079,0.132595,0.480817,0.129481,0.479379
37123,33v1,LMJ.RY0402.114699,Cre17.g720261,F08,0.222708,0.544820,0.190051,0.509158,0.266786,0.525207,...,0.164441,0.489604,0.167727,0.492650,0.185259,0.474756,0.168748,0.475247,0.185540,0.470235
37124,33v1,LMJ.RY0402.061439,Cre12.g528250,F09,0.238160,0.508602,0.234751,0.504954,0.224286,0.480517,...,0.139275,0.467601,0.158201,0.449693,0.165204,0.442341,0.141284,0.467282,0.172641,0.462803
37125,33v1,LMJ.RY0402.054236,Cre04.g215150,F10,0.325559,0.580553,0.315931,0.554926,0.273291,0.571980,...,0.241662,0.507279,0.258353,0.525903,0.231259,0.535143,0.204211,0.531585,0.248224,0.519055
37126,33v1,LMJ.RY0402.103661,Cre09.g407150,F11,0.325197,0.596757,0.333728,0.573040,0.333424,0.564433,...,0.243144,0.536050,0.266421,0.535009,0.248560,0.534424,0.263664,0.538953,0.263858,0.521353
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43365,33v3,LMJ.RY0402.174457,Cre17.g716050,F05,0.214108,0.559615,0.265574,0.522965,0.259530,0.498009,...,0.249200,0.517969,0.236174,0.512876,0.239224,0.497810,0.242151,0.470098,0.217196,0.475414
43366,33v3,LMJ.RY0402.156099,Cre16.g664850,F04,0.186296,0.502146,0.190372,0.486384,0.201537,0.500657,...,0.137000,0.449453,0.127148,0.458027,0.138138,0.433503,0.150307,0.428228,0.124546,0.439083
43367,33v3,LMJ.RY0402.115726,Cre14.g632750,F03,0.289725,0.599567,0.304983,0.588922,0.303263,0.582385,...,0.262941,0.567766,0.251345,0.545229,0.244887,0.548431,0.246925,0.532979,0.281022,0.575148
43368,33v3,LMJ.RY0402.229127,"Cre10.g431250,Cre07.g315850",F02,0.437962,0.670090,0.422470,0.639844,0.370520,0.640612,...,0.299318,0.625483,0.379764,0.585802,0.342128,0.584464,0.339103,0.579231,0.351222,0.614955


## plate 33 5min-5min

In [22]:
plates = ['33v1','33v2', '33v3']
y2_cols = [f'y2_{i}' for i in range(1, 89)]

# Run normalization
phase2_33_5min_5min_normalized = quantile_normalize_light_regime(
    df=phase2_df1,
    light_regime='5min-5min',
    plates=plates,
    y2_cols=y2_cols
)
y2_cols = [f'y2_{i}' for i in range(1, 90)]
phase2_33_5min_5min_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88,y2_89
38222,33v1,LMJ.RY0402.256825,Cre17.g697000,L02,0.139184,0.492346,0.143052,0.481995,0.116993,0.482350,...,0.421524,0.090679,0.422273,0.074850,0.442332,0.032710,0.422937,0.070378,0.416667,NaN
38223,33v1,LMJ.RY0402.145715,Cre06.g295600,L01,0.307900,0.714717,0.304738,0.691171,0.292991,0.675833,...,0.671666,0.306899,0.658406,0.274794,0.647901,0.257552,0.666384,0.272059,0.680327,NaN
38224,33v1,LMJ.RY0402.242322,Cre04.g214097,K24,0.275391,0.647616,0.274006,0.626673,0.273164,0.618863,...,0.635469,0.215263,0.651828,0.247952,0.632483,0.233822,0.652437,0.288966,0.646142,NaN
38225,33v1,LMJ.RY0402.230639,Cre07.g322550,K23,0.176045,0.576904,0.191557,0.557279,0.217448,0.556079,...,0.513764,0.164007,0.523491,0.132786,0.541604,0.138218,0.543856,0.116231,0.540710,NaN
38226,33v1,LMJ.RY0402.191679,Cre10.g433450,K22,0.191021,0.577834,0.170887,0.550621,0.207362,0.546188,...,0.524978,0.144693,0.534336,0.127646,0.544071,0.149457,0.525696,0.140805,0.516422,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44097,33v3,LMJ.RY0402.065715,"Cre01.g024800,Cre01.g040250",K20,0.239166,0.622013,0.161403,0.612789,0.183827,0.599218,...,0.573078,0.143564,0.567566,0.155285,0.581540,0.153872,0.562938,0.175514,0.553670,NaN
44098,33v3,LMJ.RY0402.191955,Cre09.g397734,K21,0.153848,0.540818,0.171319,0.542598,0.151428,0.519885,...,0.504630,0.129385,0.473630,0.067246,0.463503,0.123284,0.496718,0.015305,0.502840,NaN
44099,33v3,LMJ.RY0402.232230,Cre03.g207351,K22,0.222178,0.554258,0.211034,0.543980,0.206019,0.546430,...,0.511118,0.147428,0.468049,0.162187,0.502416,0.083178,0.506146,0.140805,0.475530,NaN
44100,33v3,LMJ.RY0402.188401,Cre06.g259950,K23,0.273558,0.637654,0.244517,0.599420,0.265948,0.635398,...,0.610304,0.177635,0.589765,0.135934,0.588895,0.197596,0.584709,0.126076,0.579838,NaN


In [23]:
plates = ['33v1', '33v2', '33v3']
phase2_33_5min_5min= phase2_df1[(phase2_df1['light_regime'] == '5min-5min') & (phase2_df1['plate'].isin(plates))].copy()
phase2_33_5min_5min[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88,y2_89
38222,33v1,LMJ.RY0402.256825,Cre17.g697000,L02,0.119654,0.483589,0.129496,0.475744,0.099980,0.473079,...,0.407433,0.077936,0.419878,0.066259,0.426068,0.004124,0.407980,0.048484,0.410037,NaN
38223,33v1,LMJ.RY0402.145715,Cre06.g295600,L01,0.296356,0.718925,0.297471,0.683677,0.278878,0.669748,...,0.663228,0.289595,0.646802,0.266496,0.641326,0.237280,0.659112,0.249374,0.675179,NaN
38224,33v1,LMJ.RY0402.242322,Cre04.g214097,K24,0.259529,0.641063,0.264846,0.617851,0.263994,0.612939,...,0.624257,0.196392,0.637665,0.229003,0.621787,0.214448,0.645196,0.262507,0.637718,NaN
38225,33v1,LMJ.RY0402.230639,Cre07.g322550,K23,0.157768,0.569068,0.180286,0.549893,0.201228,0.545440,...,0.498719,0.147969,0.510300,0.121084,0.531193,0.121830,0.532250,0.099110,0.530557,NaN
38226,33v1,LMJ.RY0402.191679,Cre10.g433450,K22,0.174487,0.569485,0.158426,0.543897,0.190466,0.537249,...,0.509453,0.127542,0.519309,0.114569,0.533412,0.133122,0.514064,0.122441,0.505592,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44097,33v3,LMJ.RY0402.065715,"Cre01.g024800,Cre01.g040250",K20,0.266058,0.628703,0.179594,0.620031,0.201475,0.608528,...,0.584152,0.166683,0.586019,0.168943,0.588543,0.177551,0.577624,0.194221,0.566249,NaN
44098,33v3,LMJ.RY0402.191955,Cre09.g397734,K21,0.184759,0.550872,0.188314,0.551771,0.167394,0.530498,...,0.517762,0.149946,0.488652,0.083168,0.473952,0.145954,0.510743,0.033668,0.512345,NaN
44099,33v3,LMJ.RY0402.232230,Cre03.g207351,K22,0.250214,0.562492,0.226988,0.553237,0.224107,0.554535,...,0.524720,0.168887,0.485601,0.174589,0.509028,0.106684,0.521802,0.162948,0.487668,NaN
44100,33v3,LMJ.RY0402.188401,Cre06.g259950,K23,0.294874,0.640702,0.261526,0.607237,0.277480,0.647085,...,0.619714,0.195490,0.604728,0.149058,0.597618,0.222482,0.596082,0.147299,0.590903,NaN


## plate 33 1min-5min

In [25]:
plates = ['33v2', '33v3']
y2_cols = [f'y2_{i}' for i in range(1, 89)]


plates_of_interest = plates  # or your list of plates
subset = phase2_df1[
    (phase2_df1['light_regime'] == '1min-5min') &
    (phase2_df1['plate'].isin(plates_of_interest))
]
mutants_by_plate = {
    plate: set(subset[subset['plate'] == plate]['mutant_ID'])
    for plate in plates_of_interest
}
common_mutants = set.intersection(*mutants_by_plate.values())
filtered_df = subset[subset['mutant_ID'].isin(common_mutants)].copy()

# Run normalization
phase2_33_1min_5min_normalized = quantile_normalize_light_regime(
    df=filtered_df,
    light_regime='1min-5min',
    plates=plates,
    y2_cols=y2_cols
)
y2_cols = [f'y2_{i}' for i in range(1, 90)]
phase2_33_1min_5min_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88,y2_89
40796,33v2,LMJ.RY0402.174457,Cre17.g716050,I05,0.139618,0.538058,0.158477,0.532758,0.147978,0.487724,...,0.533291,0.066316,0.525852,0.077919,0.510623,0.092526,0.503216,0.085533,0.544138,NaN
40797,33v2,LMJ.RY0402.079968,Cre03.g203150,K24,0.357300,0.691064,0.357062,0.681954,0.325078,0.651065,...,0.660641,0.274157,0.664892,0.249595,0.672227,0.246579,0.656960,0.259539,0.670129,NaN
40798,33v2,LMJ.RY0402.096063,Cre02.g112333,K23,0.242196,0.592794,0.204383,0.603233,0.155976,0.574992,...,0.583140,0.122198,0.600258,0.117779,0.591854,0.149647,0.599335,0.146873,0.590229,NaN
40799,33v2,LMJ.RY0402.194210,Cre09.g389578,K22,0.312081,0.681492,0.293522,0.671467,0.297828,0.692773,...,0.634004,0.219855,0.647817,0.211089,0.664688,0.218703,0.633070,0.232959,0.650161,NaN
40800,33v2,LMJ.RY0402.143718,Cre16.g648750,K21,0.354784,0.646249,0.355644,0.675022,0.311324,0.649101,...,0.631551,0.272411,0.653157,0.236307,0.621260,0.249169,0.624269,0.257991,0.625336,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43729,33v3,LMJ.RY0402.054236,Cre04.g215150,K19,0.363529,0.695742,0.308790,0.677148,0.337727,0.676088,...,0.632398,0.248682,0.656401,0.213280,0.664757,0.302141,0.683264,0.198384,0.657751,NaN
43730,33v3,LMJ.RY0402.065715,"Cre01.g024800,Cre01.g040250",K20,0.256611,0.636948,0.308436,0.614846,0.248623,0.629929,...,0.610941,0.168134,0.582071,0.147936,0.629257,0.179478,0.608905,0.209103,0.590586,NaN
43731,33v3,LMJ.RY0402.191955,Cre09.g397734,K21,0.204004,0.609427,0.204424,0.560412,0.179673,0.583381,...,0.586574,0.128825,0.531990,0.143372,0.541029,0.191700,0.552312,0.125914,0.590442,NaN
43732,33v3,LMJ.RY0402.232230,Cre03.g207351,K22,0.203578,0.572553,0.219855,0.601099,0.213973,0.579087,...,0.542165,0.116257,0.554649,0.138172,0.565167,0.172533,0.546995,0.109253,0.586394,NaN


In [26]:
plates = ['33v1', '33v2', '33v3']
phase2_33_1min_5min= phase2_df1[(phase2_df1['light_regime'] == '1min-5min') & (phase2_df1['plate'].isin(plates))].copy()
phase2_33_1min_5min[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88,y2_89
37857,33v1,LMJ.RY0402.198898,Cre04.g215800,I01,0.427137,0.706140,0.421301,0.701260,0.360474,0.723712,...,0.688250,0.369182,0.692902,0.307709,0.654141,0.350246,0.698071,0.305303,0.682960,NaN
37858,33v1,LMJ.RY0402.226911,Cre12.g513900,H23,0.288564,0.604629,0.242802,0.605002,0.250585,0.638254,...,0.643783,0.262860,0.627819,0.184237,0.628482,0.258157,0.573087,0.173733,0.595979,NaN
37859,33v1,LMJ.RY0402.049580,"Cre12.g496100,Cre12.g496150 & Cre12.g496100,Cr...",K21,0.160861,0.588692,0.260072,0.556219,0.128238,0.525060,...,0.559478,0.099411,0.561924,0.197746,0.567310,0.128583,0.547668,0.184627,0.551091,NaN
37860,33v1,LMJ.RY0402.100101,Cre02.g095058,K20,0.307926,0.653029,0.284216,0.633961,0.273934,0.620653,...,0.596428,0.200287,0.621152,0.214105,0.567864,0.197327,0.604022,0.226745,0.638897,NaN
37861,33v1,LMJ.RY0402.254836,Cre04.g215150,K18,0.327540,0.647937,0.302139,0.639456,0.318837,0.594776,...,0.619352,0.292789,0.653087,0.279688,0.624444,0.304337,0.614024,0.319898,0.642317,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43729,33v3,LMJ.RY0402.054236,Cre04.g215150,K19,0.366632,0.694560,0.311227,0.675047,0.338615,0.680897,...,0.634048,0.248322,0.657017,0.221565,0.668114,0.315752,0.683668,0.205389,0.660908,NaN
43730,33v3,LMJ.RY0402.065715,"Cre01.g024800,Cre01.g040250",K20,0.263442,0.639845,0.311175,0.616237,0.248739,0.633504,...,0.614550,0.175455,0.583748,0.155538,0.633145,0.189854,0.611222,0.214901,0.597104,NaN
43731,33v3,LMJ.RY0402.191955,Cre09.g397734,K21,0.214564,0.610416,0.211143,0.564185,0.183436,0.587020,...,0.591113,0.134932,0.536939,0.150093,0.550272,0.202568,0.557434,0.134919,0.596930,NaN
43732,33v3,LMJ.RY0402.232230,Cre03.g207351,K22,0.213877,0.576373,0.229238,0.602206,0.214685,0.583235,...,0.547423,0.123579,0.558872,0.145181,0.573318,0.182222,0.553814,0.117022,0.594767,NaN


In [27]:
phase2_33_quantile1= pd.concat([
    phase2_33_20h_ML_normalized,
    phase2_33_20h_HL_normalized,
    phase2_33_2h_2h_normalized,
    phase2_33_10min_10min_normalized,
    phase2_33_1min_1min_normalized,
    phase2_33_30s_30s_normalized,
    phase2_33_5min_5min_normalized,
    phase2_33_1min_5min_normalized
], ignore_index=True)

In [28]:
phase2_33_quantile1.to_csv('phase2_33_quantile1.csv', index= False)

In [29]:
phase2_33_quantile1.shape

(7339, 467)

In [30]:
plates = ['33v1','33v2','33v3']
phase2_df1[phase2_df1['plate'].isin(plates)].shape

(8451, 467)